done with silver....need to check for idempotency


##Flattening Nested Data

### - Customers Table

In [0]:
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, to_timestamp

products_schema = StructType([
    StructField("product_id", IntegerType()),
    StructField("product_brand", StringType()),
    StructField("product_name", StringType()),
    StructField("product_sku", LongType()),
    StructField("product_retail_price", DoubleType()),
    StructField("product_cost", DoubleType()),
    StructField("product_weight", DoubleType()),
    StructField("recyclable", IntegerType()),
    StructField("low_fat", IntegerType())
])

customers_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("customer_acct_num", LongType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("customer_address", StringType()),
    StructField("customer_city", StringType()),
    StructField("customer_state_province", StringType()),
    StructField("customer_postal_code", IntegerType()),
    StructField("customer_country", StringType()),
    StructField("birthdate", StringType()),   # convert later
    StructField("marital_status", StringType()),
    StructField("yearly_income", StringType()),
    StructField("gender", StringType()),
    StructField("total_children", IntegerType()),
    StructField("num_children_at_home", IntegerType()),
    StructField("education", StringType()),
    StructField("acct_open_date", StringType()),  # convert later
    StructField("member_card", StringType()),
    StructField("occupation", StringType()),
    StructField("homeowner", StringType())
])



In [0]:
@dlt.table(name="maven_market_uc.silver.silver_customers")
def silver_customers():

    df = dlt.read("maven_market_uc.bronze.bronze_customers")

    parsed = df.withColumn(
        "parsed",
        from_json(col("data"), customers_schema)
    )

    return (
        parsed
        .filter(col("parsed").isNotNull())               # ✅ valid JSON
        .filter(col("_fivetran_deleted") == False)       # ✅ remove deleted
        .select("parsed.*", "_fivetran_synced")

        # ✅ type corrections
        .withColumn("customer_id", col("customer_id").cast("int"))
        .withColumn("customer_acct_num", col("customer_acct_num").cast("long"))

        # ✅ date conversions
        .withColumn("birthdate", to_date("birthdate", "M/d/yyyy"))
        .withColumn("acct_open_date", to_date("acct_open_date", "M/d/yyyy"))

        # ✅ null handling (important for dimension)
        .fillna({
            "customer_city": "UNKNOWN",
            "customer_country": "UNKNOWN",
            "customer_state_province": "UNKNOWN",
            "education": "UNKNOWN",
            "occupation": "UNKNOWN"
        })

        # 🔥 correct timestamp (same as products logic)
        .withColumn("processed_time", to_timestamp(col("_fivetran_synced")))
        .drop("_fivetran_synced")

        .withColumn("_source_system", lit("fivetran"))
    )

## - Products Table

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_products")
def silver_products():

    df = dlt.read("maven_market_uc.bronze.bronze_products")

    parsed = df.withColumn(
        "parsed",
        from_json(col("data"), products_schema)
    )

    return (
        parsed
        .filter(col("parsed").isNotNull())               # ✅ valid JSON
        .filter(col("_fivetran_deleted") == False)       # ✅ remove deleted
        .select("parsed.*", "_fivetran_synced")

        # ✅ type corrections (safe casting)
        .withColumn("product_id", col("product_id").cast("int"))
        .withColumn("product_sku", col("product_sku").cast("long"))
        .withColumn("product_retail_price", col("product_retail_price").cast("double"))
        .withColumn("product_cost", col("product_cost").cast("double"))
        .withColumn("product_weight", col("product_weight").cast("double"))

        # ✅ handle null flags safely
        .withColumn("recyclable", coalesce(col("recyclable"), lit(0)).cast("int"))
        .withColumn("low_fat", coalesce(col("low_fat"), lit(0)).cast("int"))

        # 🔥 IMPORTANT: correct timestamp
        .withColumn("processed_time", to_timestamp(col("_fivetran_synced")))

        .drop("_fivetran_synced")

        # ✅ optional but recommended (consistency)
        .withColumn("_source_system", lit("fivetran"))
    )

#Cleaning of Data

Calendar

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_calendar")
def silver_calendar():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_calendar")

    return (
        df
        # ✅ convert to proper date
        .withColumn("date", to_date("date", "M/d/yyyy"))

        # ✅ remove invalid dates
        .filter(col("date").isNotNull())

        # ✅ deduplicate
        .dropDuplicates(["date"])

        # ✅ enrich (VERY USEFUL for analytics)
        .withColumn("year", year("date"))
        .withColumn("month", month("date"))
        .withColumn("day", dayofmonth("date"))
        .withColumn("day_of_week", date_format("date", "E"))
        .withColumn("week_of_year", weekofyear("date"))

        # ✅ timestamp alignment
        .withColumnRenamed("ingestion_timestamp", "processed_time")

        .withColumn("_source_system", lit("adls"))
    )

Returns

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_returns")
@dlt.expect("valid_quantity", "quantity > 0")
@dlt.expect("valid_product", "product_id IS NOT NULL")
@dlt.expect("valid_store", "store_id IS NOT NULL")
def silver_returns():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_returns")

    return (
        df
        # ✅ date conversion
        .withColumn("return_date", to_date("return_date", "M/d/yyyy"))

        # ✅ dedup (important)
        .dropDuplicates(["product_id", "store_id", "return_date"])

        # ✅ outlier flag (same pattern as transactions)
        .withColumn("is_outlier", when(col("quantity") > 500, True).otherwise(False))

        # ✅ timestamp consistency
        .withColumnRenamed("ingestion_timestamp", "processed_time")

        .withColumn("_source_system", lit("adls"))
    )

Regions Table

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_regions")
def silver_regions():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_regions")

    return (
        df
        # ✅ basic null handling (dimension data)
        .fillna({
            "sales_district": "UNKNOWN",
            "sales_region": "UNKNOWN"
        })

        # ✅ dedup (important for dimension tables)
        .dropDuplicates(["region_id"])

        # ✅ timestamp alignment
        .withColumnRenamed("ingestion_timestamp", "processed_time")

        .withColumn("_source_system", lit("adls"))
    )

-Orders Table

In [0]:
orders_schema = StructType([
    StructField("order_id", StringType()),
    StructField("product_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("store_id", IntegerType()),
    StructField("quantity", IntegerType()),
    StructField("order_date", StringType()),
    StructField("payment_type", StringType()),
    StructField("order_channel", StringType())
])

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_orders")
@dlt.expect("valid_quantity", "quantity > 0")
@dlt.expect("valid_product", "product_id IS NOT NULL")
@dlt.expect("valid_customer", "customer_id IS NOT NULL")
@dlt.expect("valid_order_date", "order_date IS NOT NULL")
def silver_orders():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_orders")

    parsed = df.withColumn(
        "parsed",
        from_json(
            col("raw_payload"),
            orders_schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_rescued_data"
            }
        )
    )

    return (
        parsed
        .filter(col("parsed").isNotNull())   # ✅ valid JSON only
        .select(
            "parsed.*",
            "_kafka_timestamp",
            "_kafka_offset",
            "_kafka_partition",
            "ingestion_timestamp"
        )

        # ✅ dedup at correct layer
        .dropDuplicates(["_kafka_partition", "_kafka_offset"])

        # ✅ transformations
        .withColumn("order_date", to_date("order_date"))
        .withColumn("payment_type", upper(trim(col("payment_type"))))
        .withColumn("is_outlier", when(col("quantity") > 500, True).otherwise(False))

        # ✅ correct timestamp
        .withColumnRenamed("ingestion_timestamp", "processed_time")

        .withColumn("_source_system", lit("kafka"))
    )

- Inventory

In [0]:
inventory_schema = StructType([
    StructField("inventory_id", StringType()),
    StructField("event_timestamp", StringType()),
    StructField("product_id", IntegerType()),
    StructField("store_id", IntegerType()),
    StructField("quantity_added", IntegerType()),
    StructField("quantity_remaining", IntegerType())
])

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_inventory")
@dlt.expect("valid_product", "product_id IS NOT NULL")
@dlt.expect("valid_store", "store_id IS NOT NULL")
def silver_inventory():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_inventory")

    parsed = df.withColumn(
        "parsed",
        from_json(
            col("raw_payload"),
            inventory_schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_rescued_data"
            }
        )
    )

    return (
        parsed
        .filter(col("parsed").isNotNull())   # ✅ valid JSON
        .select(
            "parsed.*",
            "_kafka_timestamp",
            "_kafka_partition",
            "_kafka_offset",
            "ingestion_timestamp"
        )

        # ✅ dedup (same as orders)
        .dropDuplicates(["_kafka_partition", "_kafka_offset"])

        # ✅ transformations
        .withColumn("event_timestamp", to_date("event_timestamp"))

        # ✅ correct timestamp
        .withColumnRenamed("ingestion_timestamp", "processed_time")

        .withColumn("_source_system", lit("kafka"))
    )

Stores

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_stores")
def silver_stores():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_stores")

    return (
        df
        # ✅ date parsing
        .withColumn("first_opened_date", to_date("first_opened_date", "M/d/yyyy"))
        .withColumn("last_remodel_date", to_date("last_remodel_date", "M/d/yyyy"))

        # ✅ basic null handling (dimension data)
        .fillna({
            "store_city": "UNKNOWN",
            "store_state": "UNKNOWN",
            "store_country": "UNKNOWN",
            "store_type": "UNKNOWN"
        })

        # ✅ rename timestamp (consistent with pipeline)
        .withColumnRenamed("ingestion_timestamp", "processed_time")

        .withColumn("_source_system", lit("adls"))
    )

TRANSACTIONS

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_transactions")
@dlt.expect("valid_quantity", "quantity > 0")
@dlt.expect("valid_customer", "customer_id IS NOT NULL")
@dlt.expect("valid_product", "product_id IS NOT NULL")
@dlt.expect("valid_date", "transaction_date <= current_date()")
def silver_transactions():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_transactions")

    return (
        df
        # ✅ date conversion
        .withColumn("transaction_date", to_date("transaction_date", "M/d/yyyy"))
        .withColumn("stock_date", to_date("stock_date", "M/d/yyyy"))

        # ✅ dedup
        .dropDuplicates(["product_id","customer_id","store_id","transaction_date"])

        # ✅ outlier flag
        .withColumn("is_outlier", when(col("quantity") > 500, True).otherwise(False))

        # ✅ align timestamp (IMPORTANT FIX)
        .withColumnRenamed("ingestion_timestamp", "processed_time")

        .withColumn("_source_system", lit("adls"))
    )

##Rescued Table

- Orders

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_orders_rescued")
def orders_rescued():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_orders")

    parsed = df.withColumn(
        "parsed",
        from_json(
            col("raw_payload"),
            orders_schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_rescued_data"
            }
        )
    )

    return (
        parsed
        # ✅ capture only failed parses (same logic as silver_orders inverse)
        .filter(col("parsed").isNull())

        .select(
            "raw_payload",
            "_kafka_topic",
            "_kafka_partition",
            "_kafka_offset",
            "_kafka_timestamp",
            "_ingestion_timestamp"
        )

        # ✅ consistent timestamp naming
        .withColumnRenamed("_ingestion_timestamp", "processed_time")

        .withColumn("_source_system", lit("kafka"))
        .withColumn("_source_table", lit("orders"))
    )

- Inventory 

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_inventory_rescued")
def silver_inventory_rescued():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_inventory")

    parsed = df.withColumn(
        "parsed",
        from_json(
            col("raw_payload"),
            inventory_schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_rescued_data"
            }
        )
    )

    return (
        parsed
        .filter(col("parsed").isNull())
        .select(
            "raw_payload",
            "_kafka_topic",
            "_kafka_partition",
            "_kafka_offset",
            "_kafka_timestamp",
            "_ingestion_timestamp"
        )
        .withColumnRenamed("_ingestion_timestamp", "processed_time")
        .withColumn("_source_system", lit("kafka"))
        .withColumn("_source_table", lit("inventory"))
    )

#Quarantine table

Returns

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_quarantine_returns")
def returns_quarantine():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_returns")

    return (
        df
        .withColumn(
            "error_reason",
            when(col("quantity") <= 0, "INVALID_QUANTITY")
            .when(col("product_id").isNull(), "MISSING_PRODUCT")
            .when(col("store_id").isNull(), "MISSING_STORE")
        )
        .filter(col("error_reason").isNotNull())

        .withColumnRenamed("ingestion_timestamp", "processed_time")
        .withColumn("_source_system", lit("adls"))
    )

Transactions

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_quarantine_transactions")
def transactions_quarantine():

    df = dlt.read("maven_market_uc.bronze.bronze_raw_transactions")

    return (
        df
        .withColumn(
            "error_reason",
            when(col("quantity") <= 0, "INVALID_QUANTITY")
            .when(col("product_id").isNull(), "MISSING_PRODUCT")
            .when(col("customer_id").isNull(), "MISSING_CUSTOMER")
            .when(col("transaction_date").isNull(), "MISSING_DATE")
        )
        .filter(col("error_reason").isNotNull())

        # ✅ keep useful metadata
        .withColumnRenamed("ingestion_timestamp", "processed_time")
        .withColumn("_source_system", lit("adls"))
    )

##SCD Type 2

## Products

In [0]:
dlt.create_streaming_table("maven_market_uc.silver.silver_dim_products_scd")

dlt.apply_changes(
    target="maven_market_uc.silver.silver_dim_products_scd",
    source="maven_market_uc.silver.silver_products",
    keys=["product_id"],
    sequence_by=col("processed_time"),
    stored_as_scd_type=2,
    ignore_null_updates=True
)

##Customers

In [0]:
dlt.create_streaming_table("maven_market_uc.silver_dim_customers_scd")

dlt.apply_changes(
    target="maven_market_uc.silver.silver_dim_customers_scd",
    source="maven_market_uc.silver.silver_customers",
    keys=["customer_id"],
    sequence_by=col("processed_time"),
    stored_as_scd_type=2,
    ignore_null_updates=True
)

##Stores

In [0]:
dlt.create_streaming_table("maven_market_uc.silver.silver_dim_stores_scd")

dlt.apply_changes(
    target="maven_market_uc.silver.silver_dim_stores_scd",
    source="maven_market_uc.silver.silver_stores",
    keys=["store_id"],
    sequence_by=col("processed_time"),
    stored_as_scd_type=2,
    ignore_null_updates=True
)

##FK Checks

returns

In [0]:
@dlt.table(name="maven_market_uc.silver.returns_product_fk_fail")
def returns_product_fk_fail():

    return (
        dlt.read("maven_market_uc.silver.silver_returns")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_products_scd")
            .filter(col("__IS_CURRENT") == True),
            "product_id",
            "left_anti"
        )
    )

In [0]:
@dlt.table(name="maven_market_uc.silver.returns_product_fk_pass")
def returns_product_fk_pass():

    return (
        dlt.read("maven_market_uc.silver.silver_returns")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_products_scd")
            .filter(col("__IS_CURRENT") == True),
            "product_id",
            "inner"
        )
    )

Stores

In [0]:
@dlt.table(name="maven_market_uc.silver.transactions_store_fk_fail")
def transactions_store_fk_fail():

    return (
        dlt.read("maven_market_uc.silver.silver_transactions")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_stores_scd"),
            "store_id",
            "left_anti"
        )
    )

In [0]:
@dlt.table(name="maven_market_uc.silver.transactions_store_fk_pass")
def transactions_store_fk_pass():

    return (
        dlt.read("maven_market_uc.silver.silver_transactions")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_stores_scd"),
            "store_id",
            "inner"
        )
    )

Products

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_transactions_product_fk_fail")
def transactions_product_fk_fail():

    return (
        dlt.read("maven_market_uc.silver.silver_transactions")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_products_scd"),
            "product_id",
            "left_anti"
        )
    )

In [0]:
@dlt.table(name="maven_market_uc.silver.silver_transactions_product_fk_pass")
@dlt.expect_or_drop("valid_product_fk", "product_id IS NOT NULL")
def transactions_product_fk_pass():

    return (
        dlt.read("maven_market_uc.silver.silver_transactions")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_products_scd"),
            "product_id",
            "inner"
        )
    )

customers

In [0]:
@dlt.table(name="maven_market_uc.silver.transactions_customer_fk_fail")
def transactions_customer_fk_fail():

    return (
        dlt.read("maven_market_uc.silver.silver_orders")
        .join(
            dlt.read("maven_market_uc.silver.dim_customers_scd"),
            "customer_id",
            "left_anti"
        )
    )

In [0]:
@dlt.table(name="maven_market_uc.silver.transactions_customer_fk_pass")
def transactions_customer_fk_pass():

    return (
        dlt.read("maven_market_uc.silver.silver_orders")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_customers_scd"),
            "customer_id",
            "inner"
        )
    )

In [0]:
@dlt.table(name="maven_market_uc.silver.inventory_product_fk_fail")
def inventory_product_fk_fail():

    return (
        dlt.read("maven_market_uc.silver.silver_inventory")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_products_scd"),
            "product_id",
            "left_anti"
        )
    )

In [0]:
@dlt.table(name="maven_market_uc.silver.inventory_product_fk_pass")
def inventory_product_fk_pass():

    return (
        dlt.read("maven_market_uc.silver.silver_inventory")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_products_scd"),
            "product_id",
            "inner"
        )
    )

In [0]:
@dlt.table(name="maven_market_uc.silver.transactions_customer_fk_fail")
def transactions_customer_fk_fail():

    return (
        dlt.read("maven_market_uc.silver.silver_transactions")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_customers_scd"),
            "customer_id",
            "left_anti"
        )
    )

In [0]:
@dlt.table(name="maven_market_uc.silver.transactions_customer_fk_pass")
@dlt.expect_or_drop("valid_customer_fk", "customer_id IS NOT NULL")
def transactions_customer_fk_pass():

    return (
        dlt.read("maven_market_uc.silver.silver_transactions")
        .join(
            dlt.read("maven_market_uc.silver.silver_dim_customers_scd"),
            "customer_id",
            "inner"
        )
    )

##Observability

In [0]:
@dlt.table(name="maven_market_uc.silver.audit_logs")
def audit_logs():

    df = dlt.read("maven_market_uc.silver.silver_transactions")

    return (
        df.groupBy("_source_system")
        .agg(
            count("*").alias("row_count"),
            count(when(col("quantity") <= 0, True)).alias("invalid_qty"),
            count(when(col("is_outlier") == True, True)).alias("outliers"),
            count(when(col("customer_id").isNull(), True)).alias("missing_customer"),
            count(when(col("product_id").isNull(), True)).alias("missing_product"),
        )
        .withColumn("processed_time", current_timestamp())
    )